In [ ]:
#import os
import numpy as np
from pathlib import Path
from obspy.core import UTCDateTime, read_inventory
from flovopy.enhanced.sdsclient import EnhancedSDSClient
from flovopy.processing.sam import RSAM, DSAM, VSEM
import sys
sys.path.append('../week8') # this is where set_samba_data_root.py lives
from set_samba_data_root import DATA_ROOT

DEBUG = True

# -----------------------------------------------------------------------------
# Load Response information for MVO stations (from SEISAN database)
# -----------------------------------------------------------------------------
RESPONSE_DIR = DATA_ROOT / 'SEISAN_DB' / 'CAL'
from obspy import read_inventory
stationxml = RESPONSE_DIR / 'MV.xml'
inv = read_inventory(stationxml)

# Set pre-filter for response removal. Remember this is a bandpass applied directly in the frequency domain
pre_filt = [0.1, 0.2, 18, 25]

# -----------------------------------------------------------------------------
# Output directory for RSAM (SAM) data
# -----------------------------------------------------------------------------
# Expand "~" to your home directory and create the folder if it doesn't exist
SAM_DIR = Path('~').expanduser() / 'work' / 'sam_data'
SAM_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Initialize SDS client (points to your SDS archive on disk)
# -----------------------------------------------------------------------------
SDS_MVO_ROOT = DATA_ROOT / "SDS_MVO"
mySDSclient = EnhancedSDSClient(SDS_MVO_ROOT)

# -----------------------------------------------------------------------------
# Define a source location for Soufriere Hills volcano. 
# Station distances to this lat/lon are used to "reduce" the displacement to 1 km distance.
# -----------------------------------------------------------------------------
source = {'lat':16.7164, 'lon':-62.1654}

# -----------------------------------------------------------------------------
# Define time range for processing
# -----------------------------------------------------------------------------
startTime = UTCDateTime(2003, 7, 1)   # start date (inclusive)
endTime   = UTCDateTime(2003, 7, 14)  # end date (exclusive)
taperSeconds = 60 * 60
window_seconds = 60  # window length for RSAM/DSAM/VSEM in seconds (e.g., 60 for 1-minute metrics)

# Number of seconds in one day (used for stepping through time)
secondsPerDay = 60 * 60 * 24

# Total number of days (not strictly needed, but useful for reference/debugging)
numDays = (endTime - startTime) / secondsPerDay

# Initialize loop variable
daytime = startTime

# -----------------------------------------------------------------------------
# Loop over each day and compute RSAM
# -----------------------------------------------------------------------------
while daytime < endTime:

    # -------------------------------------------------------------------------
    # Step 1: Load waveform data from SDS archive for one day
    # -------------------------------------------------------------------------
    print(f'Loading Stream data for {daytime}')

    st = mySDSclient.get_waveforms(
        "MV",     # network code
        "*",   # station code
        "*",       # location code (empty = any)
        "*Z",    # channel (vertical short-period)
        daytime - taperSeconds,
        daytime + secondsPerDay + taperSeconds
    )

    print(f'- got {len(st)} Trace ids')

    # -------------------------------------------------------------------------
    # Step 2: Pre-process the data (detrend, taper, filter)
    # -------------------------------------------------------------------------    
    for tr in st:
        tr.data = np.nan_to_num(tr.data, nan=0) # Ensure data is in float64 format for remove_response
    st.detrend('linear')  # remove linear trend
    secondsInStream = st[0].stats.endtime - st[0].stats.starttime
    st.merge(method=1, fill_value='latest')  # merge traces, filling gaps with last value
    st.detrend('demean')  # remove mean
    st.taper(max_percentage=taperSeconds/secondsInStream)  # apply a taper to the edges

    # -------------------------------------------------------------------------
    # Step 3: Compute RSAM metrics
    # -------------------------------------------------------------------------
    # RSAM = Real-time Seismic Amplitude Measurement
    # Here computed at 60-second intervals (1-minute RSAM)
    print(f'Computing RSAM metrics for {daytime}, and saving to pickle files')
    st_rsam = st.copy()
    st.taper(max_percentage=taperSeconds/secondsInStream)  # apply a taper to the edges
    st_rsam.filter('highpass', freq=0.5)
    st_rsam.trim(starttime=daytime, endtime=daytime + secondsPerDay)  # use the trimmed stream for RSAM
    if DEBUG:
        print(f'Plotting raw seismograms for {daytime}')
        st_rsam.plot();

    rsamMV24h = RSAM(stream=st_rsam,  sampling_interval=window_seconds)
    if DEBUG:
        print(f'Plotting RSAM metrics for {daytime}')
        rsamMV24h.plot();

    # -------------------------------------------------------------------------
    # Step 4: Save RSAM results to disk
    # -------------------------------------------------------------------------
    # Writes one file per station/channel/day
    rsamMV24h.write(str(SAM_DIR), ext='csv')

    # -------------------------------------------------------------------------
    # Step 5: Remove the instrument response to get displacement in meters
    # First we need to ensure the data is in a format that remove_response can handle (float64)
    # -------------------------------------------------------------------------
    st.remove_response(inventory=inv, pre_filt=pre_filt, output="DISP", taper=False, plot=True, detrend=False) 
    st.trim(starttime=daytime, endtime=daytime + secondsPerDay)
    if DEBUG:
        print(f'Plotting raw displacement seismograms for {daytime}')
        st.plot();

    # -------------------------------------------------------------------------
    # Step 6: Compute Displacement Seismic Amplitude Measurement (DSAM)
    # This is just like RSAM, but applied to a displacement seismogram
    # -------------------------------------------------------------------------
    dsamObj = DSAM(stream=st, sampling_interval=window_seconds)
    if DEBUG:
        print(f'Plotting DSAM metrics for {daytime}')
        dsamObj.plot()

    # -------------------------------------------------------------------------
    # Step 7: Compute Reduced Displacement (DR)
    # Assumes body waves (surfaceWaves=False) and no inelastic attenuation (Q=None)
    # -------------------------------------------------------------------------
    DRobj = dsamObj.compute_reduced_displacement(inv, source, surfaceWaves=False, Q=None)
    if DEBUG:
        print(f'Plotting Reduced Displacement metrics for {daytime}')
        DRobj.plot()

    # ------------------------------------------------------------
    # Step 8: Compute VSEM (Velocity Seismic Energy Measurement)
    # ------------------------------------------------------------
    st.differentiate()  # convert displacement to velocity
    vsemObj = VSEM(stream=st, sampling_interval=window_seconds)
    if DEBUG:
        print(f'Plotting VSEM metrics for {daytime}')
        vsemObj.plot(metrics='velocity')

    # ------------------------------------------------------------
    # Step 9: Compute Reduced Energy (estimate of source energy)
    # ------------------------------------------------------------    
    ERobj = vsemObj.compute_reduced_energy(inv, source, Q=None)
    if DEBUG:
        print(f'Plotting Reduced Energy metrics for {daytime}')
        ERobj.plot(metrics='energy')    

    # ------------------------------------------------------------
    # Step 10: Compute total energy and equivalent magnitude (ME)
    # ------------------------------------------------------------      
    # Compute Energy Magnitude
    E, ME = ERobj.sum_energy()

    # -------------------------------------------------------------------------
    # Step forward one day
    # -------------------------------------------------------------------------
    daytime += secondsPerDay